"This notebook performs GWAS-based signal analysis to identify significant genomic regions associated with target traits. It integrates statistical GWAS results with taglotype (haplotype) information to detect shared signals—genetic variants that are both statistically significant and structurally supported.

The workflow includes filtering significant variants, grouping them into genomic regions, evaluating taglotype support, and ranking regions based on signal strength and consistency. The output highlights the most reliable candidate regions for further biological interpretation and downstream analysis".


In [0]:
import pandas as pd
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()



In [0]:
import sys
sys.path.append("/Volumes/bmqg/default_bronze/fatemeh/final_project/modules")
import importlib
import shared_signal

importlib.reload(shared_signal)
from shared_signal import run_pipeline, get_shared_signals

In [0]:

CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

GWAS_TABLE = CONFIG["data"]["gwas_table_newharvested"]
TAGLO_TABLE = CONFIG["paths"]["TAGLO_TABLE"]

PHENO_PATH = CONFIG["paths"]["aroma_matrix_newharvested"]
pheno = pd.read_csv(PHENO_PATH)

print("Phenotype shape:", pheno.shape)



result = run_pipeline(
    spark=spark,   # 
    gwas_table=GWAS_TABLE,
    taglo_table=TAGLO_TABLE,
    pheno_df=pheno,
    phenotype_col="aroma_score"
)
display(result.head(40))


shared_signals_newhatrvested = get_shared_signals(
    spark=spark,
    gwas_table=GWAS_TABLE,
    p_thresh=1e-6
)

display(shared_signals_newhatrvested)

Phenotype shape: (95, 39)


trait,chrom,window,n_unique_pos,leader_start,leader_p,leader_nlp,taglo_id1,taglo_id2,taglo_id3,taglo_id4,n_taglo,taglo_ids,consecutive_run_len,best_consecutive_run,rank_in_trait
Benzaldehyde,ST4.03ch01,10500000,6,10800000,1.8150239589073347E-26,25.741117637761437,27159,27166,0,0,27,"List(27214, 27281, 27282, 27283, 27221, 27222, 27159, 27223, 27224, 27225, 27289, 27165, 27166, 27167, 27168, 27169, 27170, 27109, 27173, 27110, 27111, 27112, 27113, 27054, 27118, 27313, 27322)",6,"List(27165, 27166, 27167, 27168, 27169, 27170)",1
"Butanal, 3-methyl-",ST4.03ch09,5000000,10,5000000,1.74433207352396E-23,22.75837083364049,392757,392758,392759,0,91,"List(393089, 393088, 392855, 392850, 392861, 392989, 392860, 392991, 392862, 392990, 392857, 392856, 392858, 392986, 393127, 393126, 392993, 392992, 393123, 392994, 392748, 393129, 393128, 392757, 392756, 392759, 392758, 392753, 392755, 392765, 392764, 392766, 392761, 392760, 392763, 392762, 392772, 392900, 393156, 392903, 393159, 392902, 393158, 392768, 392771, 392898, 393037, 392908, 393164, 393038, 392905, 393033, 393161, 392904, 393160, 392907, 392906, 393162, 393045, 393044, 393047, 393046, 393041, 393040, 393043, 393042, 392797, 392795, 392805, 392804, 392807, 392806, 392801, 392803, 392809, 392808, 392811, 392810, 392949, 392948, 392951, 392950, 392944, 393085, 393087, 393086, 392953, 392952, 392827, 393083, 393082)",12,"List(392755, 392756, 392757, 392758, 392759, 392760, 392761, 392762, 392763, 392764, 392765, 392766)",1
"Butanal, 3-methyl-",ST4.03ch09,4500000,9,4950000,1.74433207352396E-23,22.75837083364049,392710,392712,392713,0,94,"List(392709, 392324, 392708, 392327, 392711, 392710, 392577, 392320, 392576, 392323, 392579, 392706, 392717, 392332, 392716, 392718, 392329, 392713, 392328, 392712, 392331, 392715, 392330, 392714, 392341, 392469, 392725, 392468, 392724, 392343, 392465, 392464, 392720, 392339, 392723, 392338, 392466, 392344, 392613, 392612, 392615, 392611, 392616, 392509, 392383, 392511, 392382, 392510, 392376, 392379, 392378, 392517, 392516, 392391, 392385, 392384, 392512, 392387, 392515, 392386, 392514, 392521, 392392, 392520, 392661, 392660, 392663, 392662, 392656, 392669, 392668, 392671, 392670, 392665, 392667, 392666, 392672, 392675, 392428, 392431, 392430, 392681, 392680, 392427, 392564, 392567, 392566, 392561, 392563, 392573, 392574, 392569, 392568, 392570)",11,"List(392708, 392709, 392710, 392711, 392712, 392713, 392714, 392715, 392716, 392717, 392718)",2
Benzaldehyde,ST4.03ch01,45000000,7,45200000,4.584732858528706E-22,21.338685964562522,45313,0,0,0,23,"List(45248, 45313, 45511, 45325, 45326, 45329, 45330, 45458, 45331, 45332, 45173, 45333, 45142, 45334, 45335, 45336, 45337, 45338, 45339, 45341, 45342, 45215, 45343)",11,"List(45329, 45330, 45331, 45332, 45333, 45334, 45335, 45336, 45337, 45338, 45339)",2
Benzaldehyde,ST4.03ch01,11000000,3,11250000,4.584732858528706E-22,21.338685964562522,27519,0,0,0,8,"List(27536, 27571, 27572, 27527, 27610, 27581, 27519, 27535)",2,"List(27535, 27536)",3
Benzaldehyde,ST4.03ch12,49000000,7,49300000,4.886556140063156E-21,20.310997107396176,555481,0,0,0,33,"List(555596, 555598, 555397, 555399, 555289, 555481, 555290, 555291, 555292, 555293, 555283, 555284, 555496, 555497, 555498, 555562, 555490, 555491, 555492, 555493, 555494, 555495, 555559, 555449, 555322, 555451, 555324, 555326, 555505, 555569, 555506, 555315, 555507)",9,"List(555490, 555491, 555492, 555493, 555494, 555495, 555496, 555497, 555498)",4
"Butanal, 3-methyl-",ST4.03ch09,2000000,9,2150000,1.0648420102487008E-19,18.972714823368005,390199,0,0,0,92,"List(390149, 390148, 390151, 390150, 390406, 390145, 390144, 390157, 390156, 390158, 390153, 390409, 390152, 390155, 390154, 390293, 390164, 390295, 390161, 390163, 390162, 390296, 390299, 390305, 390433, 390306, 390189, 390445, 390447, 390190, 390446, 390184, 390312, 390442, 390453, 390196, 390452, 390199, 390455, 390198, 390454, 390448, 390195, 390451, 390450, 390333, 390460, 390335, 390201, 390457, 390456, 390459, 390202

trait,chrom,start,p_wald,taglo_id1,taglo_id2,taglo_id3,taglo_id4,taglo_id,nlp
Dimethyl disulfide,ST4.03ch08,11700000,5.461004E-7,355759,355766,0,0,355759,6.262727497932459
Dimethyl disulfide,ST4.03ch08,11700000,5.461004E-7,355759,355766,0,0,355766,6.262727497932459
Dimethyl disulfide,ST4.03ch08,11800000,5.461004E-7,355910,355914,0,0,355910,6.262727497932459
Dimethyl disulfide,ST4.03ch08,11800000,5.461004E-7,355910,355914,0,0,355914,6.262727497932459
Dimethyl disulfide,ST4.03ch11,40600000,4.198617E-7,519810,0,0,0,519810,6.376893748312966
Dimethyl disulfide,ST4.03ch11,39900000,7.888862E-8,519274,519275,0,0,519274,7.102985650328006
Dimethyl disulfide,ST4.03ch11,39900000,7.888862E-8,519274,519275,0,0,519275,7.102985650328006
Dimethyl disulfide,ST4.03ch08,42300000,6.95852E-7,378115,378116,378118,378121,378115,6.157483114629232
Dimethyl disulfide,ST4.03ch08,42300000,6.95852E-7,378115,378116,378118,378121,378116,6.157483114629232
Dimethyl disulfide,ST4.03ch08,42300000,6.95852E-7,378115,378116,378118,378121,378118,6.157483114629232
